In [1]:
# Imports
import numpy as np 
import copy as cp

# Add the given value to the total score
def add_score(sc, val):
    sc += val
    return sc

# Shift the grid left, merge tiles and update the score
def move_left(grid, score):
    for i in range(4):
        non_zero = [x for x in grid[i,:] if x != 0]
        zero = [0] * (4 - len(non_zero))
        grid[i,:] = np.array(non_zero + zero)
        for j in range(3):
            if grid[i,j] == grid[i,j+1]:
                grid[i,j] *= 2
                score = add_score(score, grid[i,j])
                grid[i,j+1] = 0
        non_zero = [x for x in grid[i,:] if x != 0]
        zero = [0] * (4 - len(non_zero))
        grid[i,:] = np.array(non_zero + zero)
    return (grid,score)

# Shift the grid right, merge tiles and update the score
def move_right(grid, score):
    for i in range(4):
        non_zero = [x for x in grid[i,:] if x != 0]
        zero = [0] * (4 - len(non_zero))
        grid[i,:] = np.array(zero + non_zero[::-1])
        for j in range(3, 0, -1):
            if grid[i,j] == grid[i,j-1]:
                grid[i,j] *= 2
                score = add_score(score, grid[i,j])
                grid[i,j-1] = 0
        non_zero = [x for x in grid[i,:] if x != 0]
        zero = [0] * (4 - len(non_zero))
        grid[i,:] = np.array(zero + non_zero[::-1])
    return (grid,score)

# Shift the grid up, merge tiles and update the score
def move_up(grid, score):
    for i in range(4):
        non_zero = [x for x in grid[:,i] if x != 0]
        zero = [0] * (4 - len(non_zero))
        grid[:,i] = np.array(non_zero + zero)
        for j in range(3):
            if grid[j,i] == grid[j+1,i]:
                grid[j,i] *= 2
                score = add_score(score, grid[j,i])
                grid[j+1,i] = 0
        non_zero = [x for x in grid[:,i] if x != 0]
        zero = [0] * (4 - len(non_zero))
        grid[:,i] = np.array(non_zero + zero)
    return (grid,score)

# Shift the grid down, merge tiles and update the score
def move_down(grid, score):
    for i in range(4):
        non_zero = [x for x in grid[:,i] if x != 0]
        zero = [0] * (4 - len(non_zero))
        grid[:,i] = np.array(zero + non_zero[::-1])
        for j in range(3, 0, -1):
            if grid[j,i] == grid[j-1,i]:
                grid[j,i] *= 2
                score = add_score(score, grid[j,i])
                grid[j-1,i] = 0
        non_zero = [x for x in grid[:,i] if x != 0]
        zero = [0] * (4 - len(non_zero))
        grid[:,i] = np.array(zero + non_zero[::-1])
    return (grid,score)

# Spawn a new tile (2 with probability 0.9, otherwise 4) on a random empty cell
def add_new_number(grid):
    zero_indices = np.where(grid == 0)
    if len(zero_indices[0]) == 0:
        return False
    index = np.random.choice(len(zero_indices[0]))
    i, j = zero_indices[0][index], zero_indices[1][index]
    grid[i,j] = 2 if np.random.random() < 0.9 else 4
    return True

# Return True if no move is possible (game over)
def check_game_over(grid):
    
    if np.all(grid) == False:
        return False
    
    for row in range(4):
        for col in range(4):
            if row != 3:
                if (grid[row,col]==grid[row+1,col]):
                    return False
            if col != 3:
                if (grid[row,col]==grid[row,col+1]):
                    return False
            
    return True

# Return True if the 2048 tile has been reached
def check_win(grid):
    return 2048 in grid

# Apply a move in the given direction and check for a win or loss.
# Raises RuntimeError("GO") if the game is over (no moves left).
# Raises RuntimeError("WIN") if the move creates a 2048 tile.
def play_2048(grid, move, score):
    
    orig_grid = cp.deepcopy(grid)
    
    if check_game_over(grid):
        raise RuntimeError("GO")
        
    if move == 'left':
        grid, score = move_left(grid, score)
    elif move == 'right':
        grid, score = move_right(grid, score)
    elif move == 'up':
        grid, score = move_up(grid, score)
    elif move == 'down':
        grid, score = move_down(grid, score)
    else:
        raise ValueError("Invalid move")
   
    if check_win(grid):
        raise RuntimeError("WIN")

    # Spawn a new tile only if the move changed the grid
    if np.array_equal(grid,orig_grid) == False:
        add_new_number(grid)
    return (grid,score)

# Start a new game: two random tiles and a score of 0
def new_game():
    score = 0
    grid = np.zeros((4,4), dtype=int)
    add_new_number(grid)
    add_new_number(grid)
    
    return (grid, score)

# Print the grid and the current score
def print_grid(grid, score):
    print('Score: ', score)
    print("+----+----+----+----+")
    for i in range(4):
        line = "|"
        for j in range(4):
            if grid[i,j] == 0:
                line += "    |"
            else:
                line += "{:4d}|".format(grid[i,j])
        print(line)
        print("+----+----+----+----+")

In [2]:
# Baseline: play uniformly random moves
grid, score = new_game()
for i in range(1000):
    direction = np.random.choice(('left','right','up','down'))
    try:
        grid, score = play_2048(grid, direction, score)
    except RuntimeError as inst:
        if(str(inst)=="GO"):
            print("GAME OVER in ",(i+1)," moves")
        elif(str(inst)=="WIN"):
            print("WIN in ",(i+1)," moves")
        break
print_grid(grid, score)

GAME OVER in  192  moves
Score:  1460
+----+----+----+----+
|   2|  32|   4|   2|
+----+----+----+----+
|  16|  64| 128|   4|
+----+----+----+----+
|   2|   8|  32|  16|
+----+----+----+----+
|   4|  16|   8|   2|
+----+----+----+----+


# Greedy Agent

In [19]:
def greedy_move(grid, score):
    """
    Pick a move with a one-step greedy policy.
    Simulates every possible move and chooses the one with the highest immediate
    score gain, breaking ties by the number of empty cells.
    """
    moves = ['left', 'right', 'up', 'down']
    best_move = None
    best_score = -1
    best_empty = -1
    original_score = score

    for move in moves:
        # Copy the grid and score for simulation
        new_grid = cp.deepcopy(grid)
        new_score = cp.copy(score)
        
        # Simulate the move
        if move == 'left':
            new_grid, new_score = move_left(new_grid, new_score)
        elif move == 'right':
            new_grid, new_score = move_right(new_grid, new_score)
        elif move == 'up':
            new_grid, new_score = move_up(new_grid, new_score)
        elif move == 'down':
            new_grid, new_score = move_down(new_grid, new_score)

        # Skip moves that do not change the grid
        if np.array_equal(new_grid, grid):
            continue

        # Heuristic: score gain, then number of empty cells
        score_gain = new_score - original_score
        empty_cells = np.count_nonzero(new_grid == 0)

        # Keep the best move so far
        if (score_gain > best_score) or (score_gain == best_score and empty_cells > best_empty):
            best_score = score_gain
            best_empty = empty_cells
            best_move = move

    return best_move if best_move is not None else np.random.choice(moves)


In [35]:
# Single game played by the greedy agent
grid, score = new_game()
for i in range(1000):
    direction = greedy_move(grid, score)  # greedy agent instead of a random move
    try:
        grid, score = play_2048(grid, direction, score)
    except RuntimeError as inst:
        if str(inst) == "GO":
            print("GAME OVER in", (i + 1), "moves")
        elif str(inst) == "WIN":
            print("WIN in", (i + 1), "moves")
        break
print_grid(grid, score)

GAME OVER in 220 moves
Score:  2752
+----+----+----+----+
|   4|   2|   4|   2|
+----+----+----+----+
|  16| 256| 128|  16|
+----+----+----+----+
|   4|  32|   4|   8|
+----+----+----+----+
|   2|   4|  16|   4|
+----+----+----+----+


In [38]:
results = []
for game_index in range(30):
    grid, score = new_game()
    moves_count = 0
    max_tile = 0

    for i in range(1000):
        direction = greedy_move(grid, score)
        try:
            grid, score = play_2048(grid, direction, score)
            moves_count += 1
            max_tile = max(max_tile, grid.max())
        except RuntimeError as inst:
            if str(inst) == "GO":
                results.append({"score": score, "moves": moves_count, "max_tile": max_tile, "win": False})
                break
            elif str(inst) == "WIN":
                results.append({"score": score, "moves": moves_count, "max_tile": max_tile, "win": True})
                break

if results:
    scores = [result["score"] for result in results]
    moves = [result["moves"] for result in results]
    max_tiles = [result["max_tile"] for result in results]
    wins = sum(result["win"] for result in results)
    losses = len(results) - wins

    statistics = {
        "best_score": max(scores),
        "worst_score": min(scores),
        "average_score": sum(scores) / len(scores),
        "wins": wins,
        "losses": losses,
        "average_max_tile": sum(max_tiles) / len(max_tiles),
        "average_moves": sum(moves) / len(moves)
    }

    # Print the statistics line by line and remove np.int64
    for key, value in statistics.items():
        print(f"{key}: {value.item() if isinstance(value, np.int64) else value}")
else:
    print("No games were recorded. Debugging required.")

best_score: 7084
worst_score: 1024
average_score: 3335.6
wins: 0
losses: 30
average_max_tile: 260.26666666666665
average_moves: 273.76666666666665


# Weighted Heuristic Agent

In [172]:
def calculate_smoothness(grid):
    """Penalize large differences between neighbouring tiles (log2 scale)."""
    smoothness = 0
    log_grid = np.log2(grid + (grid == 0))  # avoid log(0)
    
    for i in range(4):
        for j in range(4):
            if i < 3:
                smoothness -= (log_grid[i, j] - log_grid[i+1, j]) ** 2  # squared penalty
            if j < 3:
                smoothness -= (log_grid[i, j] - log_grid[i, j+1]) ** 2
    
    return smoothness


def calculate_monotonicity(grid):
    """Reward boards whose values decrease monotonically along rows and columns."""
    monotonicity = 0
    log_grid = np.log2(grid + (grid == 0))
    
    for i in range(4):
        for j in range(3):
            if log_grid[i, j] > log_grid[i, j+1]:
                monotonicity += log_grid[i, j] - log_grid[i, j+1]
    
    for j in range(4):
        for i in range(3):
            if log_grid[i, j] > log_grid[i+1, j]:
                monotonicity += log_grid[i, j] - log_grid[i+1, j]
    
    return monotonicity


def edge_bonus(grid):
    """Positional bonus for large tiles near the top-left corner."""
    edge_weights = np.array([
        [6, 5, 4, 3],
        [5, 4, 3, 2],
        [4, 3, 2, 1],
        [3, 2, 1, 0]
    ])
    
    return np.sum(grid * edge_weights)

def count_available_moves(grid):
    """Number of legal moves (higher is better)."""
    count = 0
    for move in ['left', 'right', 'up', 'down']:
        new_grid = cp.deepcopy(grid)
        new_score = 0  # score is irrelevant here; we only check whether the move changes the grid
        
        if move == 'left':
            new_grid, _ = move_left(new_grid, new_score)
        elif move == 'right':
            new_grid, _ = move_right(new_grid, new_score)
        elif move == 'up':
            new_grid, _ = move_up(new_grid, new_score)
        elif move == 'down':
            new_grid, _ = move_down(new_grid, new_score)
        
        if not np.array_equal(new_grid, grid):
            count += 1
    
    return count


def max_tile_corner_bonus(grid):
    """Return a bonus if the largest tile sits in one of the corners (defined but not used by heuristic_move)."""
    max_tile = np.max(grid)
    if grid[0, 0] == max_tile or grid[0, 3] == max_tile or grid[3, 0] == max_tile or grid[3, 3] == max_tile:
        return 100  # corner-strategy bonus
    return 0



def heuristic_move(grid, score):
    moves = ['left', 'right', 'up', 'down']
    best_move = None
    best_heuristic = -float('inf')
    original_score = score

    for move in moves:
        new_grid = cp.deepcopy(grid)
        new_score = cp.copy(score)

        # Simulate the move
        if move == 'left':
            new_grid, new_score = move_left(new_grid, new_score)
        elif move == 'right':
            new_grid, new_score = move_right(new_grid, new_score)
        elif move == 'up':
            new_grid, new_score = move_up(new_grid, new_score)
        elif move == 'down':
            new_grid, new_score = move_down(new_grid, new_score)

        if np.array_equal(new_grid, grid):
            continue  # move did not change the grid; ignore it

        # Compute heuristic features
        score_gain = new_score - original_score
        empty_cells = np.count_nonzero(new_grid == 0)
        smoothness = calculate_smoothness(new_grid)
        monotonicity = calculate_monotonicity(new_grid)
        move_count = count_available_moves(new_grid)
        corner_bonus = edge_bonus(new_grid)

        # Weighted combination of the features
        heuristic = (
            1.0 * score_gain +
            2.0 * empty_cells +
            3.0 * smoothness +
            2.0 * monotonicity +
            1.5 * move_count +
            3.0 * corner_bonus
        )

        if heuristic > best_heuristic:
            best_heuristic = heuristic
            best_move = move

    return best_move if best_move is not None else np.random.choice(moves)


In [174]:
grid, score = new_game()
for i in range(1000):
    direction = heuristic_move(grid, score)  # heuristic agent instead of a random move
    try:
        grid, score = play_2048(grid, direction, score)
    except RuntimeError as inst:
        if str(inst) == "GO":
            print("GAME OVER in", (i + 1), "moves")
        elif str(inst) == "WIN":
            print("WIN in", (i + 1), "moves")
        break
print_grid(grid, score)

GAME OVER in 304 moves
Score:  3556
+----+----+----+----+
| 256| 128|  64|   4|
+----+----+----+----+
|  64|  32|  16|   8|
+----+----+----+----+
|  32|  16|   8|   4|
+----+----+----+----+
|  16|   8|   4|   2|
+----+----+----+----+


In [180]:
results = []
for game_index in range(30):
    grid, score = new_game()
    moves_count = 0
    max_tile = 0

    for i in range(1000):
        direction = heuristic_move(grid, score)
        try:
            grid, score = play_2048(grid, direction, score)
            moves_count += 1
            max_tile = max(max_tile, grid.max())
        except RuntimeError as inst:
            if str(inst) == "GO":
                results.append({"score": score, "moves": moves_count, "max_tile": max_tile, "win": False})
                break
            elif str(inst) == "WIN":
                results.append({"score": score, "moves": moves_count, "max_tile": max_tile, "win": True})
                break

if results:
    scores = [result["score"] for result in results]
    moves = [result["moves"] for result in results]
    max_tiles = [result["max_tile"] for result in results]
    wins = sum(result["win"] for result in results)
    losses = len(results) - wins

    statistics = {
        "best_score": max(scores),
        "worst_score": min(scores),
        "average_score": sum(scores) / len(scores),
        "wins": wins,
        "losses": losses,
        "average_max_tile": sum(max_tiles) / len(max_tiles),
        "average_moves": sum(moves) / len(moves)
    }

    # Print the statistics line by line and remove np.int64
    for key, value in statistics.items():
        print(f"{key}: {value.item() if isinstance(value, np.int64) else value}")
else:
    print("No games were recorded. Debugging required.")

best_score: 6640
worst_score: 740
average_score: 2784.9333333333334
wins: 0
losses: 30
average_max_tile: 211.2
average_moves: 243.63333333333333


# Basic Expectimax (fixed depth 4)

In [181]:
def evaluate_grid(grid):
    """Heuristic evaluation of a grid."""
    return np.sum(grid) + 10 * np.count_nonzero(grid == 0)  # simple heuristic: tile sum + empty-cell bonus

def expectimax(grid, depth, player_turn):
    """Depth-limited expectimax search for 2048."""
    
    if depth == 0:
        return evaluate_grid(grid), None  # leaf: return heuristic value, no move

    if player_turn:
        # Max node: the player picks the best move
        best_move = None
        best_value = -float('inf')

        for move in ['left', 'right', 'up', 'down']:
            new_grid = cp.deepcopy(grid)
            new_score = 0  

            if move == 'left':
                new_grid, new_score = move_left(new_grid, new_score)
            elif move == 'right':
                new_grid, new_score = move_right(new_grid, new_score)
            elif move == 'up':
                new_grid, new_score = move_up(new_grid, new_score)
            elif move == 'down':
                new_grid, new_score = move_down(new_grid, new_score)

            if np.array_equal(new_grid, grid):
                continue  # move did not change the grid

            value, _ = expectimax(new_grid, depth - 1, player_turn=False)

            if value > best_value:
                best_value = value
                best_move = move

        return best_value, best_move

    else:
        # Chance node: the game spawns a 2 or a 4
        empty_cells = np.argwhere(grid == 0)
        if len(empty_cells) == 0:
            return evaluate_grid(grid), None  # no empty cells

        total_expectation = 0
        for (x, y) in empty_cells:
            test_grid = grid.copy()
            
            # Spawn a 2 (probability 0.9)
            test_grid[x, y] = 2
            value_2, _ = expectimax(test_grid, depth - 1, player_turn=True)
            
            # Spawn a 4 (probability 0.1)
            test_grid[x, y] = 4
            value_4, _ = expectimax(test_grid, depth - 1, player_turn=True)

            total_expectation += 0.9 * value_2 + 0.1 * value_4

        return total_expectation / len(empty_cells), None


In [182]:
# Play one fully automatic game
grid, score = new_game()

for i in range(1000):  # at most 1000 moves
    _, direction = expectimax(grid, depth=4, player_turn=True)  # agent chooses a move

    if direction is None:
        print("GAME OVER in", (i + 1), "moves")
        break

    try:
        grid, score = play_2048(grid, direction, score)  # apply the move
    except RuntimeError as inst:
        if str(inst) == "GO":
            print("GAME OVER in", (i + 1), "moves")
        elif str(inst) == "WIN":
            print("WIN in", (i + 1), "moves")
        break

# Final game state
print("Final state:")
print_grid(grid, score)

GAME OVER in 384 moves
Final state:
Score:  5448
+----+----+----+----+
|   2|  32|  16|   4|
+----+----+----+----+
|   4| 128|   8|   2|
+----+----+----+----+
|  64|  16| 512|   4|
+----+----+----+----+
|   2|   8|  32|   4|
+----+----+----+----+


In [183]:
# Benchmark: expectimax over 30 games
results = []

for game_index in range(30):  # 30 games
    grid, score = new_game()  # start a new game
    moves_count = 0
    max_tile = 0

    for i in range(1000):  # at most 1000 moves per game
        _, direction = expectimax(grid, depth=4, player_turn=True)  # agent chooses a move

        if direction is None:  # no legal move: the game is over
            results.append({"score": score, "moves": moves_count, "max_tile": max_tile, "win": False})
            print(f"GAME OVER in {moves_count} moves with score {score}")
            break

        try:
            grid, score = play_2048(grid, direction, score)  # apply the move
            moves_count += 1
            max_tile = max(max_tile, grid.max())  # track the largest tile
        except RuntimeError as inst:
            if str(inst) == "GO":  # game lost
                results.append({"score": score, "moves": moves_count, "max_tile": max_tile, "win": False})
                print(f"GAME OVER in {moves_count} moves with score {score}")
                break
            elif str(inst) == "WIN":  # game won (2048 tile reached)
                results.append({"score": score, "moves": moves_count, "max_tile": max_tile, "win": True})
                print(f"WIN in {moves_count} moves with score {score}")
                break

# Summary statistics for expectimax
if results:
    scores = [result["score"] for result in results]
    moves = [result["moves"] for result in results]
    max_tiles = [result["max_tile"] for result in results]
    wins = sum(result["win"] for result in results)
    losses = len(results) - wins

    statistics = {
        "best_score": max(scores),
        "worst_score": min(scores),
        "average_score": sum(scores) / len(scores),
        "wins": wins,
        "losses": losses,
        "average_max_tile": sum(max_tiles) / len(max_tiles),
        "average_moves": sum(moves) / len(moves)
    }

    # Print the statistics (converting np.int64 to int)
    print("\n📈 Expectimax Statistics:")
    for key, value in statistics.items():
        print(f"{key}: {value.item() if isinstance(value, np.int64) else value}")
else:
    print("No games were recorded. Debugging required.")


GAME OVER in 829 moves with score 14336
GAME OVER in 378 moves with score 5456
GAME OVER in 785 moves with score 13784
GAME OVER in 426 moves with score 6328
GAME OVER in 407 moves with score 6056
GAME OVER in 576 moves with score 8528
GAME OVER in 480 moves with score 7060
GAME OVER in 651 moves with score 11340
GAME OVER in 386 moves with score 5484
GAME OVER in 717 moves with score 12232
GAME OVER in 483 moves with score 7072
GAME OVER in 730 moves with score 12324
GAME OVER in 446 moves with score 6776
GAME OVER in 697 moves with score 11896
GAME OVER in 439 moves with score 6424
GAME OVER in 459 moves with score 6700
GAME OVER in 521 moves with score 7484
GAME OVER in 544 moves with score 7780
GAME OVER in 779 moves with score 13680
GAME OVER in 913 moves with score 15836
GAME OVER in 726 moves with score 12200
GAME OVER in 283 moves with score 3404
GAME OVER in 715 moves with score 11972
GAME OVER in 380 moves with score 5548
GAME OVER in 423 moves with score 6336
GAME OVER in 46

# Expectimax with an Improved Heuristic and Dynamic Depth

In [188]:
def evaluate_grid(grid):
    """Improved heuristic evaluation of a grid."""
    # Sum of tile values
    sum_value = np.sum(grid)

    # Number of empty cells
    empty_cells = np.count_nonzero(grid == 0)

    # Bonus for large tiles in the corners
    corner_bonus = 0
    corner_bonus += grid[0, 0] * 0.1
    corner_bonus += grid[0, 3] * 0.1
    corner_bonus += grid[3, 0] * 0.1
    corner_bonus += grid[3, 3] * 0.1

    # Smoothness: number of equal adjacent tile pairs (possible merges)
    smoothness = 0
    for i in range(4):
        for j in range(4):
            if i < 3 and grid[i, j] == grid[i + 1, j]:
                smoothness += 1
            if j < 3 and grid[i, j] == grid[i, j + 1]:
                smoothness += 1

    # Weighted combination
    return sum_value + 10 * empty_cells + corner_bonus + 10 * smoothness


def simulate_tile_addition(grid):
    """Enumerate all possible tile spawns (2 or 4) on empty cells with their probabilities."""
    empty_cells = np.argwhere(grid == 0)
    if len(empty_cells) == 0:
        return []  # no empty cells, nothing to simulate

    simulations = []
    for cell in empty_cells:
        i, j = cell
        # Raise the probability of a 4 in later stages of the game
        for tile_value, probability in [(2, 0.9), (4, 0.1)]:
            # Adjust the spawn probabilities based on the board total
            if np.sum(grid) > 1000:  # later stage of the game
                probability = 0.2 if tile_value == 4 else 0.8
            
            new_grid = cp.deepcopy(grid)
            new_grid[i][j] = tile_value
            simulations.append((new_grid, probability))
    
    return simulations




def expectimax_depth(grid, depth, is_maximizing_player=True):
    if depth == 0 or check_game_over(grid):
        return evaluate_grid(grid), None

    if is_maximizing_player:  # Max node: the player picks the best move
        best_value = -float('inf')
        best_move = None

        for move in ['left', 'right', 'up', 'down']:
            new_grid = cp.deepcopy(grid)
            new_score = 0

            # Apply the corresponding move
            if move == 'left':
                new_grid, new_score = move_left(new_grid, new_score)
            elif move == 'right':
                new_grid, new_score = move_right(new_grid, new_score)
            elif move == 'up':
                new_grid, new_score = move_up(new_grid, new_score)
            elif move == 'down':
                new_grid, new_score = move_down(new_grid, new_score)

            if np.array_equal(new_grid, grid):
                continue  # move did not change the grid

            value, _ = expectimax_depth(new_grid, depth - 1, False)  # continue with the environment's (chance) turn

            if value > best_value:
                best_value = value
                best_move = move

        return best_value, best_move

    else:  # Chance node: the environment spawns a new tile
        simulations = simulate_tile_addition(grid)
        if not simulations:
            return evaluate_grid(grid), None

        total_value = 0
        for sim_grid, probability in simulations:
            value, _ = expectimax_depth(sim_grid, depth - 1, True)
            total_value += probability * value

        return total_value / len(simulations), None


In [192]:
grid, score = new_game()

for i in range(1000):  # at most 1000 moves
    # Increase the search depth as the game progresses
    if i < 200:
        depth = 2
    elif i < 400:
        depth = 3
    elif i < 600:
        depth = 4
    else:
        depth = 5

    _, direction = expectimax_depth(grid, depth, is_maximizing_player=True)  # agent chooses a move

    if direction is None:
        print("GAME OVER in", (i + 1), "moves")
        break

    try:
        grid, score = play_2048(grid, direction, score)  # apply the move
    except RuntimeError as inst:
        if str(inst) == "GO":
            print("GAME OVER in", (i + 1), "moves")
        elif str(inst) == "WIN":
            print("WIN in", (i + 1), "moves")
        break

# Final game state
print("Final state:")
print_grid(grid, score)

GAME OVER in 509 moves
Final state:
Score:  7364
+----+----+----+----+
|   2|   8|  64| 512|
+----+----+----+----+
|   4| 128|  16|   2|
+----+----+----+----+
|   8|  16|  64|   8|
+----+----+----+----+
|   2|  32|   2| 256|
+----+----+----+----+


In [186]:
# Benchmark: expectimax with dynamic depth over 30 games.
# Games that reach the 1000-move cap without a win or game over are not recorded.
results = []

for game_index in range(30):  # 30 games
    grid, score = new_game()  # start a new game
    moves_count = 0
    max_tile = 0

    for i in range(1000):  # at most 1000 moves per game
        # Increase the search depth as the game progresses
        if i < 200:
            depth = 2
        elif i < 400:
            depth = 3
        elif i < 600:
            depth = 4
        else:
            depth = 5

        _, direction = expectimax_depth(grid, depth, is_maximizing_player=True)  # agent chooses a move

        if direction is None:  # no legal move: the game is over
            results.append({"score": score, "moves": moves_count, "max_tile": max_tile, "win": False})
            print(f"GAME OVER in {moves_count} moves with score {score}")
            break

        try:
            grid, score = play_2048(grid, direction, score)  # apply the move
            moves_count += 1
            max_tile = max(max_tile, grid.max())  # track the largest tile
        except RuntimeError as inst:
            if str(inst) == "GO":  # game lost
                results.append({"score": score, "moves": moves_count, "max_tile": max_tile, "win": False})
                print(f"GAME OVER in {moves_count} moves with score {score}")
                break
            elif str(inst) == "WIN":  # game won (2048 tile reached)
                results.append({"score": score, "moves": moves_count, "max_tile": max_tile, "win": True})
                print(f"WIN in {moves_count} moves with score {score}")
                break


# Summary statistics for expectimax with dynamic depth
if results:
    scores = [result["score"] for result in results]
    moves = [result["moves"] for result in results]
    max_tiles = [result["max_tile"] for result in results]
    wins = sum(result["win"] for result in results)
    losses = len(results) - wins

    statistics = {
        "best_score": max(scores),
        "worst_score": min(scores),
        "average_score": sum(scores) / len(scores),
        "wins": wins,
        "losses": losses,
        "average_max_tile": sum(max_tiles) / len(max_tiles),
        "average_moves": sum(moves) / len(moves)
    }

    # Print the statistics (converting np.int64 to int)
    print("\n📈 Expectimax Statistics with Dynamic Depth:")
    for key, value in statistics.items():
        print(f"{key}: {value.item() if isinstance(value, np.int64) else value}")
else:
    print("No games were recorded. Debugging required.")


GAME OVER in 723 moves with score 12152
GAME OVER in 981 moves with score 16720
GAME OVER in 616 moves with score 9504
GAME OVER in 478 moves with score 7040
GAME OVER in 470 moves with score 6900
GAME OVER in 509 moves with score 8264
GAME OVER in 759 moves with score 12516
GAME OVER in 492 moves with score 7152
GAME OVER in 409 moves with score 5732
GAME OVER in 512 moves with score 7500
GAME OVER in 649 moves with score 10112
GAME OVER in 473 moves with score 7024
GAME OVER in 558 moves with score 7948
GAME OVER in 741 moves with score 12276
GAME OVER in 883 moves with score 15444
GAME OVER in 499 moves with score 7216
GAME OVER in 495 moves with score 7164
GAME OVER in 918 moves with score 15676
GAME OVER in 400 moves with score 5604
GAME OVER in 702 moves with score 11928
GAME OVER in 933 moves with score 15952
GAME OVER in 474 moves with score 7008
GAME OVER in 830 moves with score 14344
GAME OVER in 480 moves with score 7112
GAME OVER in 568 moves with score 8756
GAME OVER in 55

## Improved Expectimax (monotonicity and smoothness evaluation)

In [198]:
def evaluate_board(grid):
    empty_tiles = np.sum(grid == 0)
    max_tile = np.max(grid)
    monotonicity = calculate_monotonicity(grid)
    smoothness = calculate_smoothness(grid)

    # Weighted score
    return (
        2 * empty_tiles + 
        4 * max_tile + 
        1 * monotonicity + 
        1 * smoothness
    )

# Note: these two functions replace the log-scale versions defined for the
# heuristic agent; here they work on raw tile values.
def calculate_monotonicity(grid):
    score = 0
    for row in grid:
        for i in range(3):
            if row[i] > row[i + 1]:
                score -= abs(row[i] - row[i + 1])
    for col in grid.T:
        for i in range(3):
            if col[i] > col[i + 1]:
                score -= abs(col[i] - col[i + 1])
    return score

def calculate_smoothness(grid):
    score = 0
    for row in grid:
        for i in range(3):
            if row[i] == row[i + 1]:
                score += row[i]
    for col in grid.T:
        for i in range(3):
            if col[i] == col[i + 1]:
                score += col[i]
    return score


# Expectimax search (returns the expected value only).
# Note: simulated moves go through play_2048, which also spawns a random tile and
# raises RuntimeError("WIN") when a simulated move creates a 2048 tile. That
# exception is not caught here, so the game loop records a win as soon as the
# search reaches 2048, which can be a move or two before it appears on the board.
def expectimax_good(grid, depth, is_maximizing_player):
    if depth == 0 or check_game_over(grid):
        return evaluate_board(grid)

    if is_maximizing_player:
        max_eval = -np.inf
        for move in ['left', 'right', 'up', 'down']:
            temp_grid = cp.deepcopy(grid)
            try:
                temp_grid, _ = play_2048(temp_grid, move, score=0)
                eval_score = expectimax_good(temp_grid, depth - 1, False)
                max_eval = max(max_eval, eval_score)
            except ValueError:
                continue
        return max_eval
    else:
        zero_indices = list(zip(*np.where(grid == 0)))
        if not zero_indices:
            return evaluate_board(grid)

        total_score = 0
        for i in zero_indices:
            temp_grid_2 = cp.deepcopy(grid)
            temp_grid_2[i] = 2
            total_score += 0.9 * expectimax_good(temp_grid_2, depth - 1, True)

            temp_grid_4 = cp.deepcopy(grid)
            temp_grid_4[i] = 4
            total_score += 0.1 * expectimax_good(temp_grid_4, depth - 1, True)

        return total_score / len(zero_indices)

def find_best_move_with_expectimax(grid):
    best_move = None
    max_eval = -np.inf

    for move in ['left', 'right', 'up', 'down']:
        temp_grid = cp.deepcopy(grid)
        try:
            temp_grid, _ = play_2048(temp_grid, move, score=0)
            if not np.array_equal(temp_grid, grid):  # only consider moves that change the grid
                eval_score = expectimax_good(temp_grid, depth=3, is_maximizing_player=False)  # increased depth
                if eval_score > max_eval:
                    max_eval = eval_score
                    best_move = move
        except ValueError:
            continue

    return best_move


In [195]:
# Expectimax agent with cycle detection: one game
grid, score = new_game()
max_moves = 1000

for i in range(max_moves):
    try:
        best_move = find_best_move_with_expectimax(grid)

        # No legal move found (should only happen at game over): stop
        if best_move is None:
            print("No valid move found. Game over.")
            break

        # Apply the best move
        orig_grid = cp.deepcopy(grid)  # keep the previous grid for cycle detection
        grid, score = play_2048(grid, best_move, score)

        # If the grid did not change after the move (a cycle), pick a random move.
        # Note: the random move is selected but not applied in this loop.
        if np.array_equal(orig_grid, grid):
            print("Cycle detected. Performing random move.")
            best_move = np.random.choice(['left', 'right', 'up', 'down'])


        # Print the grid every 100 moves
        if (i + 1) % 100 == 0:
            print(f"Move {i + 1}:")
            print_grid(grid, score)

    except RuntimeError as inst:
        if str(inst) == "GO":
            print("GAME OVER in", (i + 1), "moves")
        elif str(inst) == "WIN":
            print("WIN in", (i + 1), "moves")
        break

# Print the final grid and score
print("Final state:")
print_grid(grid, score)

Move 100:
Score:  1024
+----+----+----+----+
|   4|   2|   8|   2|
+----+----+----+----+
|    |    |   8|  32|
+----+----+----+----+
|    |    |    |  32|
+----+----+----+----+
|   2|   2|   4| 128|
+----+----+----+----+
Move 200:
Score:  2368
+----+----+----+----+
|   2|   4|   8|   8|
+----+----+----+----+
|   8|   2|   4|  16|
+----+----+----+----+
|   2|  16|  16|  64|
+----+----+----+----+
|   2|    |  32| 256|
+----+----+----+----+
Move 300:
Score:  4572
+----+----+----+----+
|    |    |    |   4|
+----+----+----+----+
|    |    |    |  16|
+----+----+----+----+
|   2|  32|  64|  16|
+----+----+----+----+
|    |   4|  16| 512|
+----+----+----+----+
Move 400:
Score:  6200
+----+----+----+----+
|    |   2|    |    |
+----+----+----+----+
|    |   2|   4|  64|
+----+----+----+----+
|    |   4|  32| 256|
+----+----+----+----+
|    |    |   8| 512|
+----+----+----+----+
Move 500:
Score:  9244
+----+----+----+----+
|    |    |   2|   4|
+----+----+----+----+
|    |   2|  16|   4|
+----

In [ ]:
# Benchmark: improved expectimax over 30 games.
# Games that reach the 1000-move cap without a win or game over are not recorded.
num_games = 30
max_moves = 1000
results = []

for game_index in range(num_games):
    grid, score = new_game()
    moves_count = 0
    max_tile = 0

    for i in range(max_moves):
        try:
            best_move = find_best_move_with_expectimax(grid)

            if best_move is None:
                print(f"[Game {game_index + 1}] No valid move found. Game over.")
                break

            # Cycle detection
            orig_grid = cp.deepcopy(grid)
            grid, score = play_2048(grid, best_move, score)

            if np.array_equal(orig_grid, grid):
                print(f"[Game {game_index + 1}] Cycle detected. Performing random move.")
                best_move = np.random.choice(['left', 'right', 'up', 'down'])

            moves_count += 1
            max_tile = max(max_tile, grid.max())

        except RuntimeError as inst:
            if str(inst) == "GO":
                print(f"[Game {game_index + 1}] GAME OVER in {moves_count} moves.")
                results.append({"score": score, "moves": moves_count, "max_tile": max_tile, "win": False})
            elif str(inst) == "WIN":
                print(f"[Game {game_index + 1}] WIN in {moves_count} moves.")
                results.append({"score": score, "moves": moves_count, "max_tile": max_tile, "win": True})
            break

# Summary statistics
if results:
    scores = [result["score"] for result in results]
    moves = [result["moves"] for result in results]
    max_tiles = [result["max_tile"] for result in results]
    wins = sum(result["win"] for result in results)
    losses = len(results) - wins

    statistics = {
        "best_score": max(scores),
        "worst_score": min(scores),
        "average_score": sum(scores) / len(scores),
        "wins": wins,
        "losses": losses,
        "average_max_tile": sum(max_tiles) / len(max_tiles),
        "average_moves": sum(moves) / len(moves)
    }

    # Print the statistics
    print("\n=== Expectimax Statistics ===")
    for key, value in statistics.items():
        print(f"{key}: {value}")
else:
    print("No games were recorded. Debugging required.")


[Game 1] WIN in 981 moves.
[Game 2] WIN in 988 moves.
[Game 3] WIN in 996 moves.
[Game 4] GAME OVER in 285 moves.
[Game 5] GAME OVER in 316 moves.
[Game 6] GAME OVER in 571 moves.
[Game 7] GAME OVER in 512 moves.
[Game 8] WIN in 975 moves.
[Game 9] GAME OVER in 434 moves.
[Game 10] GAME OVER in 503 moves.
[Game 11] WIN in 942 moves.
[Game 12] GAME OVER in 450 moves.
[Game 14] WIN in 986 moves.
[Game 15] GAME OVER in 954 moves.
[Game 16] WIN in 943 moves.
[Game 17] GAME OVER in 921 moves.
[Game 18] GAME OVER in 906 moves.
[Game 19] WIN in 982 moves.
[Game 20] GAME OVER in 941 moves.
[Game 21] GAME OVER in 521 moves.
[Game 22] WIN in 951 moves.
[Game 23] GAME OVER in 992 moves.
[Game 24] WIN in 999 moves.
[Game 26] GAME OVER in 563 moves.
[Game 28] WIN in 943 moves.
[Game 29] GAME OVER in 536 moves.
[Game 30] GAME OVER in 552 moves.

=== Expectimax Statistics ===
best_score: 17560
worst_score: 3440
average_score: 13022.37037037037
wins: 11
losses: 16
average_max_tile: 862.8148148148148
a